In [ ]:
import gc
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from collections import deque

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms


# ============================================================
# 0) Utilities
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def cleanup() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@torch.no_grad()
def accuracy(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return float(correct / total) if total > 0 else 0.0


def mean_deque(d: deque) -> float:
    return float(np.mean(list(d))) if len(d) > 0 else float("nan")


def fmt_table7(df: pd.DataFrame) -> str:
    # Table-style 4 decimals
    df2 = df.copy()
    for c in df2.columns:
        if pd.api.types.is_numeric_dtype(df2[c]):
            df2[c] = df2[c].map(lambda v: "–" if pd.isna(v) else f"{v:.4f}")
    return df2.to_string(index=False)


# ============================================================
# 1) Table 7 config (paper notation)
# ============================================================

@dataclass
class Table7Config:
    # FL (Colab-friendly)
    K: int = 12          # clients
    R: int = 25          # rounds
    E: int = 1           # local epochs
    B: int = 64          # batch size
    eta: float = 0.01    # learning rate
    mu: float = 0.9      # momentum

    # Non-IID
    alpha_dir: float = 0.5

    # Head/Tail split
    head: Tuple[int, ...] = (0, 1, 2, 3, 4)
    tail: Tuple[int, ...] = (5, 6, 7, 8, 9)

    # Public leak (for head-only gaming)
    phi: float = 1.0

    # Strategy mix (same skeleton as E2/E3)
    theta_benign: float = 0.10
    theta_gaming: float = 0.30
    s_update: float = 1.75

    # Alignment knob: M_pub(λ) = (1-λ) M_head + λ M_tail
    lambdas: Tuple[float, ...] = (0.00, 0.30, 0.60)

    # Summary over last L rounds
    L: int = 7

    seed: int = 42
    root: str = "./data"
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


CFG = Table7Config()


# ============================================================
# 2) Data prep (Fashion-MNIST)
#   - 50k local partitioned
#   - last 10k public candidates -> public head loader + public tail loader
#   - leak subset: head-only from public candidates
#   - per-client tail-eval loader from heldout local eval
# ============================================================

def make_dirichlet_partitions(labels: np.ndarray, K: int, alpha_dir: float, rng: np.random.Generator) -> List[List[int]]:
    n_classes = int(labels.max()) + 1
    client_indices: List[List[int]] = [[] for _ in range(K)]

    class_indices = []
    for k in range(n_classes):
        idx_k = np.where(labels == k)[0]
        rng.shuffle(idx_k)
        class_indices.append(idx_k)

    for k in range(n_classes):
        idx_k = class_indices[k]
        n_k = len(idx_k)
        if n_k == 0:
            continue
        proportions = rng.dirichlet(alpha_dir * np.ones(K))
        proportions = proportions / proportions.sum()
        splits = (np.cumsum(proportions) * n_k).astype(int)
        shards = np.split(idx_k, splits[:-1])
        for cid, shard in enumerate(shards):
            client_indices[cid].extend(shard.tolist())

    for cid in range(K):
        rng.shuffle(client_indices[cid])

    return client_indices


def filter_subset_by_labels(
    base_subset: Subset,
    train_local_set: Subset,
    full_train: datasets.FashionMNIST,
    allowed: Tuple[int, ...],
) -> Subset:
    allowed_set = set(allowed)
    kept = []
    for idx_in_local in base_subset.indices:
        global_idx = train_local_set.indices[idx_in_local]
        y = int(full_train.targets[global_idx])
        if y in allowed_set:
            kept.append(idx_in_local)
    return Subset(train_local_set, kept)


def split_subset_train_eval(base_subset: Subset, seed: int, eval_ratio: float = 0.2) -> Tuple[Subset, Subset]:
    idxs = list(base_subset.indices)
    rng = np.random.default_rng(seed)
    rng.shuffle(idxs)
    n_eval = int(round(eval_ratio * len(idxs)))
    ev = idxs[:n_eval]
    tr = idxs[n_eval:]
    return Subset(base_subset.dataset, tr), Subset(base_subset.dataset, ev)


def oversample_tail(
    train_subset: Subset,
    train_local_set: Subset,
    full_train: datasets.FashionMNIST,
    tail: Tuple[int, ...],
    factor: int = 2,
) -> ConcatDataset:
    tail_sub = filter_subset_by_labels(train_subset, train_local_set, full_train, tail)
    if len(tail_sub) == 0:
        return ConcatDataset([train_subset])
    return ConcatDataset([train_subset] + [tail_sub for _ in range(factor)])


def prepare_data(cfg: Table7Config):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    full_train = datasets.FashionMNIST(root=cfg.root, train=True, download=True, transform=transform)

    # 60k -> 50k local + 10k public candidates
    n_local = 50_000
    idx_all = np.arange(len(full_train))
    idx_local = idx_all[:n_local]
    idx_public = idx_all[n_local:]

    train_local_set = Subset(full_train, idx_local.tolist())
    targets_all = np.array(full_train.targets)

    head_mask = np.isin(targets_all, np.array(cfg.head, dtype=int))
    tail_mask = np.isin(targets_all, np.array(cfg.tail, dtype=int))

    public_head_idx = np.array([i for i in idx_public if head_mask[i]], dtype=int)
    public_tail_idx = np.array([i for i in idx_public if tail_mask[i]], dtype=int)

    rng = np.random.default_rng(cfg.seed)
    rng.shuffle(public_head_idx)
    rng.shuffle(public_tail_idx)

    public_head_set = Subset(full_train, public_head_idx.tolist())
    public_tail_set = Subset(full_train, public_tail_idx.tolist())

    public_head_loader = DataLoader(public_head_set, batch_size=cfg.B, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
    public_tail_loader = DataLoader(public_tail_set, batch_size=cfg.B, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    # leak subset (head-only public)
    leak_size = int(cfg.phi * len(public_head_idx))
    leak_idx = public_head_idx[:leak_size]
    leak_subset = Subset(full_train, leak_idx.tolist())

    # dirichlet partitions on local 50k
    train_targets_local = np.array(full_train.targets)[idx_local]
    client_indices = make_dirichlet_partitions(
        labels=train_targets_local,
        K=cfg.K,
        alpha_dir=cfg.alpha_dir,
        rng=np.random.default_rng(cfg.seed)
    )
    client_base = [Subset(train_local_set, idxs) for idxs in client_indices]

    # per-client train split + tail-eval loader
    client_train: List[Subset] = []
    client_tail_eval_loaders: List[Optional[DataLoader]] = []

    for cid in range(cfg.K):
        tr, ev = split_subset_train_eval(client_base[cid], seed=cfg.seed + 123 + cid, eval_ratio=0.2)
        client_train.append(tr)

        tail_ev = filter_subset_by_labels(ev, train_local_set, full_train, cfg.tail)
        if len(tail_ev) == 0:
            client_tail_eval_loaders.append(None)
        else:
            client_tail_eval_loaders.append(
                DataLoader(tail_ev, batch_size=cfg.B, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
            )

    cleanup()
    return full_train, train_local_set, client_train, client_tail_eval_loaders, public_head_loader, public_tail_loader, leak_subset


# ============================================================
# 3) Model + local training
# ============================================================

class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def make_model(device: torch.device) -> nn.Module:
    return SimpleCNN().to(device)


def local_train(model: nn.Module, dataset, cfg: Table7Config) -> None:
    loader = DataLoader(dataset, batch_size=cfg.B, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
    opt = optim.SGD(model.parameters(), lr=cfg.eta, momentum=cfg.mu)
    crit = nn.CrossEntropyLoss()

    model.train()
    for _ in range(cfg.E):
        for x, y in loader:
            x = x.to(cfg.device, non_blocking=True)
            y = y.to(cfg.device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x), y)
            loss.backward()
            opt.step()


# ============================================================
# 4) Strategies (same as E2/E3)
# ============================================================

STR_HONEST = "HONEST"
STR_BENIGN = "BENIGN"
STR_GAME_HEAD = "GAME_HEAD"
STR_GAME_SCALE = "GAME_SCALE"


def assign_strategies(cfg: Table7Config, theta_gaming: float, rng: np.random.Generator) -> Dict[int, str]:
    ids = np.arange(cfg.K)
    rng.shuffle(ids)

    n_benign = int(round(cfg.theta_benign * cfg.K))
    n_gaming = int(round(theta_gaming * cfg.K))

    benign_ids = ids[:n_benign]
    gaming_ids = ids[n_benign:n_benign + n_gaming]
    honest_ids = ids[n_benign + n_gaming:]

    strat: Dict[int, str] = {int(cid): STR_HONEST for cid in honest_ids}
    for cid in benign_ids:
        strat[int(cid)] = STR_BENIGN

    half = len(gaming_ids) // 2
    for cid in gaming_ids[:half]:
        strat[int(cid)] = STR_GAME_HEAD
    for cid in gaming_ids[half:]:
        strat[int(cid)] = STR_GAME_SCALE

    return strat


def build_client_train_dataset(
    cid: int,
    strat: str,
    cfg: Table7Config,
    base_train: Subset,
    full_train,
    train_local_set,
    leak_subset,
):
    if strat == STR_HONEST:
        return base_train
    if strat == STR_BENIGN:
        return oversample_tail(base_train, train_local_set, full_train, cfg.tail, factor=2)
    if strat == STR_GAME_HEAD:
        head_only = filter_subset_by_labels(base_train, train_local_set, full_train, cfg.head)
        return ConcatDataset([head_only, leak_subset])
    if strat == STR_GAME_SCALE:
        return base_train
    raise ValueError(strat)


# ============================================================
# 5) One run: streaming FedAvg, keep only last-L for Table 7
# ============================================================

def run_once(
    cfg: Table7Config,
    data_bundle,
    theta_gaming: float,
    lam: float,
    seed_offset: int,
) -> Tuple[float, float]:
    """
    Returns:
      M_bar: mean_{last L} M_pub(λ)
      W_bar: mean_{last L} welfare W (mean tail acc across clients)
    """
    full_train, train_local_set, client_train, client_tail_eval_loaders, public_head_loader, public_tail_loader, leak_subset = data_bundle

    set_seed(cfg.seed + seed_offset)
    rng = np.random.default_rng(cfg.seed + seed_offset)

    strat_map = assign_strategies(cfg, theta_gaming=theta_gaming, rng=rng)

    client_train_ds = []
    for cid in range(cfg.K):
        ds = build_client_train_dataset(
            cid=cid,
            strat=strat_map[cid],
            cfg=cfg,
            base_train=client_train[cid],
            full_train=full_train,
            train_local_set=train_local_set,
            leak_subset=leak_subset,
        )
        client_train_ds.append(ds)

    # global state on CPU
    model0 = make_model(cfg.device)
    global_sd_cpu = {k: v.detach().to("cpu").clone() for k, v in model0.state_dict().items()}
    del model0
    cleanup()

    M_last = deque(maxlen=cfg.L)
    W_last = deque(maxlen=cfg.L)

    for _ in range(cfg.R):
        sum_delta = {k: torch.zeros_like(v, device="cpu") for k, v in global_sd_cpu.items()}

        for cid in range(cfg.K):
            local_model = make_model(cfg.device)
            local_model.load_state_dict(global_sd_cpu, strict=True)

            local_train(local_model, client_train_ds[cid], cfg)
            local_sd = local_model.state_dict()

            scale = cfg.s_update if (strat_map[cid] == STR_GAME_SCALE and theta_gaming > 0.0) else 1.0

            # delta on CPU
            with torch.no_grad():
                for k in global_sd_cpu.keys():
                    sum_delta[k] += scale * (local_sd[k].detach().to("cpu") - global_sd_cpu[k])

            del local_model, local_sd
            cleanup()

        # global update: global + avg(delta)
        with torch.no_grad():
            for k in global_sd_cpu.keys():
                global_sd_cpu[k] = global_sd_cpu[k] + (sum_delta[k] / float(cfg.K))

        del sum_delta
        cleanup()

        # Evaluate M_head, M_tail, M_pub(λ), and W
        eval_model = make_model(cfg.device)
        eval_model.load_state_dict(global_sd_cpu, strict=True)

        M_head = accuracy(eval_model, public_head_loader, cfg.device)
        M_tail = accuracy(eval_model, public_tail_loader, cfg.device)
        M_pub = (1.0 - lam) * M_head + lam * M_tail

        tail_accs = []
        for cid in range(cfg.K):
            loader = client_tail_eval_loaders[cid]
            if loader is None:
                tail_accs.append(np.nan)
            else:
                tail_accs.append(accuracy(eval_model, loader, cfg.device))
        W = float(np.nanmean(np.array(tail_accs, dtype=float)))

        M_last.append(M_pub)
        W_last.append(W)

        del eval_model
        cleanup()

    return mean_deque(M_last), mean_deque(W_last)


# ============================================================
# 6) Table 7: λ sweep, aligned vs gaming summary per λ
# ============================================================

def run_table_7(cfg: Table7Config) -> pd.DataFrame:
    set_seed(cfg.seed)
    data_bundle = prepare_data(cfg)

    rows = []
    for lam in cfg.lambdas:
        # deterministic offsets per λ and condition
        off_aligned = 1000 + int(round(100 * lam))
        off_gaming = 2000 + int(round(100 * lam))

        M_a, W_a = run_once(cfg, data_bundle, theta_gaming=0.0, lam=float(lam), seed_offset=off_aligned)
        M_g, W_g = run_once(cfg, data_bundle, theta_gaming=cfg.theta_gaming, lam=float(lam), seed_offset=off_gaming)

        gap_a = M_a - W_a
        gap_g = M_g - W_g
        dW = W_a - W_g
        dgap = gap_g - gap_a
        pog_paired = np.nan if (not np.isfinite(W_a) or W_a <= 1e-12) else (W_a - W_g) / W_a

        rows.append({
            "lambda": float(lam),
            "M_aligned": float(M_a),
            "W_aligned": float(W_a),
            "M_gaming": float(M_g),
            "W_gaming": float(W_g),
            "Delta_W": float(dW),
            "Delta_gap": float(dgap),
            "PoG_paired": float(pog_paired),
        })

    del data_bundle
    cleanup()

    return pd.DataFrame(rows).sort_values("lambda").reset_index(drop=True)


if __name__ == "__main__":
    df7 = run_table_7(CFG)
    print("Table 7: High-alignment regime via mixed public metric M_pub(lambda).")
    print(fmt_table7(df7))